In [3]:
import os, tinker
from transformers import AutoTokenizer
from tinker_cookbook.checkpoint_utils import get_last_checkpoint

MODEL = "meta-llama/Llama-3.1-8B-Instruct"
LOG_PATH = "/Users/moe/Documents/ML Projects/Tinker-Project/Tinker-Projects/results/run-smoke"
LORA_RANK = 32

# 1) Letzten Checkpoint finden
ckpt = get_last_checkpoint(LOG_PATH)  # liest checkpoints.jsonl
if not ckpt or "state_path" not in ckpt:
    raise FileNotFoundError("Kein gespeicherter State. Lauf ggf. neu mit kurzer Config oder warte bis ein Save erfolgt.")
STATE_PATH = ckpt["state_path"]

# 2) Training- und Sampling-Client
svc = tinker.ServiceClient()
training = svc.create_lora_training_client(base_model=MODEL, rank=LORA_RANK)
training.load_state(STATE_PATH).result()
sampling = training.save_weights_and_get_sampling_client(name="nb-sample")

# 3) Sampling (Chat-Template)
tok = AutoTokenizer.from_pretrained(MODEL, use_fast=True)
messages = [{"role":"user","content":"Gib mir eine kurze, präzise Erklärung von DPO in 3 Sätzen."}]
text = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
input_ids = tok.encode(text)

out = sampling.sample(
    prompt=tinker.ModelInput.from_ints(input_ids),
    sampling_params=tinker.SamplingParams(max_tokens=128, temperature=0.7, top_p=0.9),
    num_samples=2
).result()

for i, seq in enumerate(out.sequences):
    print(f"{i}: {tok.decode(seq.tokens[len(input_ids):], skip_special_tokens=True).strip()}\n")

FileNotFoundError: Kein gespeicherter State. Lauf ggf. neu mit kurzer Config oder warte bis ein Save erfolgt.

In [4]:
rest = svc.create_rest_client()
archive = rest.download_checkpoint_archive_from_tinker_path(sampling.model_path).result()
with open("model-checkpoint.tar.gz", "wb") as f:
    f.write(archive)

AttributeError: 'RestClient' object has no attribute 'download_checkpoint_archive_from_tinker_path'